# 11. SVM + SBERT + 판정요지 하이브리드 재순위화

05번의 Linear SVM Top-3 후보 생성 구조를 유지하면서 HS 지식베이스 검색과 과거 판정요지 기반 사례 검색을 결합합니다.

- A: TF-IDF 지식베이스 재순위화
- B: SBERT 지식베이스 재순위화
- C: TF-IDF 지식베이스 + SBERT 판정요지 사례 검색
- D: SBERT 지식베이스 + SBERT 판정요지 사례 검색

신규 제품 질의에는 제품 속성 5개만 사용합니다. 과거 사례 문서에는 학습 Fold의 제품 속성 5개와 전처리 판정요지를 사용합니다. 평가 Fold의 판정요지와 정답 HS 코드는 검색 및 재순위화에 사용하지 않습니다.

모든 방법은 동일한 SVM Top-3 후보 안에서 순위를 조정합니다. 제외 규칙은 긍정 점수에 포함하지 않고 별도 경고 근거로 보존합니다. 본 실험은 의사결정 지원을 위한 탐색적 평가이며 법적 품목분류를 확정하지 않습니다.

In [ ]:
!pip -q install -U sentence-transformers

In [ ]:
import re, hashlib
import numpy as np
import pandas as pd
from urllib.parse import quote
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import MinMaxScaler

## 1. 데이터 불러오기 및 전처리

In [ ]:
BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
cases_file = quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
data_all = pd.read_csv(BASE+cases_file).fillna('')
kb = pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
input_cols = ['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
target_col,summary_col = 'HS Code','Preprocessed Ruling Summary'
required_cols = [target_col]+input_cols+[summary_col]
missing = [c for c in required_cols if c not in data_all.columns]
if missing: raise ValueError(f'필수 컬럼 누락: {missing}')
data_all[required_cols] = data_all[required_cols].astype(str).replace('nan','').fillna('')
data_all[target_col] = data_all[target_col].str.strip()
data_all['query_text'] = data_all[input_cols].agg(' '.join,axis=1).str.strip()
data_all['case_doc'] = (data_all['query_text']+' '+data_all[summary_col]).str.strip()
data_all['duplicate_group'] = data_all[input_cols].astype(str).agg('||'.join,axis=1).map(lambda x:hashlib.md5(x.encode()).hexdigest())
eligible = data_all.groupby(target_col).filter(lambda x:len(x)>=4)
group_counts = eligible.groupby(target_col)['duplicate_group'].nunique()
labels = sorted(group_counts[group_counts>=4].index.astype(str))
data = eligible[eligible[target_col].isin(labels)].copy().reset_index(drop=True)
data[target_col] = data[target_col].astype(str)
kb['document_text'] = kb['document_text'].fillna('').astype(str)
kb['hs_code'] = kb['hs_code'].fillna('').astype(str)
kb['search_text'] = (kb['document_text']+' '+kb['rule_type'].astype(str)+' '+kb['source_sheet'].astype(str)).str.strip()
print('제품:',len(data),'| HS 코드:',data[target_col].nunique(),'| KB 문서:',len(kb),'| 판정요지 결측:',int(data[summary_col].str.strip().eq('').sum()))
display(data[[target_col]+input_cols+[summary_col]].head(3))

## 2. 규칙 유형 및 검색기 설정

규칙의 시트명만으로 제외 규칙을 판정하지 않습니다. 제외 여부는 명시적인 rule_type에 근거합니다. 상위·하위 HS 코드 호환성은 숫자 prefix로 확인합니다.

In [ ]:
def code_digits(value): return re.sub(r'\D','',str(value))
def code_compatible(rule_code,candidate_code):
    a,b = code_digits(rule_code),code_digits(candidate_code)
    return bool(a and b and (a.startswith(b) or b.startswith(a)))
def is_exclusion(row): return str(row.get('rule_type','')).strip() in {'제외','제외·예외'}
def is_positive_rule(row): return (not is_exclusion(row)) and str(row.get('rule_type','')).strip() in {'포함','소호별 포함 범위','소호별 포함 범위 및 제외','조건부 포함','조건부 분류','분류원칙·예외'}
kb['is_exclusion'] = kb.apply(is_exclusion,axis=1)
kb['is_positive'] = kb.apply(is_positive_rule,axis=1)
kb['code_digits'] = kb['hs_code'].map(code_digits)
print('긍정 규칙:',int(kb['is_positive'].sum()),'| 제외 규칙:',int(kb['is_exclusion'].sum()))
MODEL_NAME = 'intfloat/multilingual-e5-base'
model = SentenceTransformer(MODEL_NAME)
print('SBERT:',MODEL_NAME,'| device:',model.device)
kb_embeddings = model.encode(['passage: '+x for x in kb['search_text']],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
tfidf_kb = FeatureUnion([('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),max_features=30000,sublinear_tf=True)),('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),max_features=30000,sublinear_tf=True))])
kb_tfidf_matrix = tfidf_kb.fit_transform(kb['search_text'])
def kb_scores_tfidf(query): return (kb_tfidf_matrix@tfidf_kb.transform([query]).T).toarray().ravel()
def kb_scores_sbert(query):
    q = model.encode(['query: '+query],normalize_embeddings=True,convert_to_numpy=True)
    return (kb_embeddings@q[0]).astype(float)
def candidate_kb_evidence(scores,candidate,top_k=30):
    idx = np.argsort(scores)[::-1][:top_k]
    docs = kb.iloc[idx].copy()
    docs['similarity'] = scores[idx]
    docs['code_match'] = docs['hs_code'].map(lambda x:code_compatible(x,candidate))
    pos = docs[docs['is_positive']&docs['code_match']]
    exc = docs[docs['is_exclusion']]
    score = float(pos['similarity'].max()) if len(pos) else 0.0
    return score,pos.head(3),exc.head(3)

## 3. 비교 실험 설계

SVM은 Fold별 학습 데이터로 학습하고, 평가 제품별 Top-3 후보를 생성합니다. 후보 3개의 SVM decision score는 후보 내에서 0~1로 정규화합니다. KB 및 사례 cosine similarity는 0~1 범위로 변환해 결합합니다. 모든 비교군에 같은 결합식을 적용하고 검색 엔진과 사례 근거의 유무만 바꿉니다.

In [ ]:
METHODS = ['A_TFIDF_KB','B_SBERT_KB','C_TFIDF_KB_SBERT_CASE','D_SBERT_KB_SBERT_CASE']
ALPHA_KB,ALPHA_CASE = 0.20,0.20
N_SPLITS,TOP_CANDIDATES = 4,3
cv = StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=42)
def scale_candidate_scores(values):
    values = np.asarray(values,dtype=float)
    if len(values)==1 or np.ptp(values)==0: return np.ones(len(values))
    return MinMaxScaler().fit_transform(values.reshape(-1,1)).ravel()
def scale_similarity(value): return float(np.clip(value,0,1))

## 4. Fold별 학습·검색·재순위화

과거 사례 임베딩은 각 Fold의 학습 사례로만 생성합니다. 신규 제품 질의는 제품 속성 5개뿐이며 평가 제품의 판정요지는 참조하지 않습니다. 지식베이스는 독립적인 가이드북 자료입니다.

In [ ]:
oof_rows,evidence_rows,fold_rows = [],[],[]
for fold,(tr,te) in enumerate(cv.split(data['query_text'],data[target_col],groups=data['duplicate_group']),1):
    train,test = data.iloc[tr].copy(),data.iloc[te].copy()
    overlap = len(set(train['duplicate_group'])&set(test['duplicate_group']))
    fold_rows.append({'fold':fold,'train_n':len(train),'test_n':len(test),'group_overlap':overlap})
    vec = FeatureUnion([('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1,max_features=30000,sublinear_tf=True)),('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=30000,sublinear_tf=True))])
    Xtr,Xte = vec.fit_transform(train['query_text']),vec.transform(test['query_text'])
    clf = LinearSVC(C=1,class_weight='balanced',max_iter=5000)
    clf.fit(Xtr,train[target_col])
    classes = clf.classes_
    train_labels = train[target_col].astype(str).to_numpy()
    case_doc_emb = model.encode(['passage: '+x for x in train['case_doc'].tolist()],batch_size=32,show_progress_bar=False,normalize_embeddings=True,convert_to_numpy=True)
    query_embs = model.encode(['query: '+x for x in test['query_text'].tolist()],batch_size=32,show_progress_bar=False,normalize_embeddings=True,convert_to_numpy=True)
    decision = clf.decision_function(Xte)
    if decision.ndim==1: decision=np.column_stack([-decision,decision])
    for local_i,(_,row) in enumerate(test.iterrows()):
        order = np.argsort(decision[local_i])[::-1][:TOP_CANDIDATES]
        candidates = [str(classes[j]) for j in order]
        svm_scaled = scale_candidate_scores(decision[local_i,order])
        qemb = query_embs[local_i]
        kb_tfidf = kb_scores_tfidf(row['query_text'])
        kb_sbert = kb_scores_sbert(row['query_text'])
        case_sims = case_doc_emb@qemb
        method_scores = {m:[] for m in METHODS}
        for pos,code in enumerate(candidates):
            tf_score,tf_docs,tf_exc = candidate_kb_evidence(kb_tfidf,code)
            sb_score,sb_docs,sb_exc = candidate_kb_evidence(kb_sbert,code)
            case_mask = train_labels==code
            case_idx = np.where(case_mask)[0]
            case_idx = case_idx[np.argsort(case_sims[case_idx])[::-1]][:3] if len(case_idx) else []
            case_score = scale_similarity(float(case_sims[case_idx[0]])) if len(case_idx) else 0.0
            for method in METHODS:
                use_sb_kb = method in {'B_SBERT_KB','D_SBERT_KB_SBERT_CASE'}
                use_case = method in {'C_TFIDF_KB_SBERT_CASE','D_SBERT_KB_SBERT_CASE'}
                kb_score = sb_score if use_sb_kb else tf_score
                method_scores[method].append(float(svm_scaled[pos]+ALPHA_KB*scale_similarity(kb_score)+(ALPHA_CASE*case_score if use_case else 0)))
            for kind,docs in [('tfidf_positive',tf_docs),('sbert_positive',sb_docs),('tfidf_exclusion',tf_exc),('sbert_exclusion',sb_exc)]:
                for _,d in docs.iterrows():
                    evidence_rows.append({'fold':fold,'row_index':int(row.name),'true_hs':row[target_col],'candidate_hs':code,'evidence_kind':kind,'knowledge_id':d['knowledge_id'],'rule_hs_code':d['hs_code'],'rule_type':d['rule_type'],'similarity':float(d['similarity']),'document_text':d['document_text']})
            for rank,idx in enumerate(case_idx,1):
                evidence_rows.append({'fold':fold,'row_index':int(row.name),'true_hs':row[target_col],'candidate_hs':code,'evidence_kind':'sbert_case_ruling','retrieved_row_index':int(train.index[idx]),'retrieved_hs':train_labels[idx],'similarity':float(case_sims[idx]),'ruling_summary':train.iloc[idx][summary_col]})
        result = {'fold':fold,'row_index':int(row.name),'true_hs':row[target_col],'svm_top1':candidates[0],'svm_top3':' | '.join(candidates)}
        for method in METHODS:
            ranked = sorted(zip(candidates,method_scores[method]),key=lambda x:x[1],reverse=True)
            result[method+'_top1'] = ranked[0][0]
            result[method+'_top3'] = ' | '.join(x[0] for x in ranked)
        oof_rows.append(result)
    print(f'Fold {fold}: train={len(train)}, test={len(test)}, overlap={overlap}')
oof = pd.DataFrame(oof_rows)
evidence = pd.DataFrame(evidence_rows)
fold_audit = pd.DataFrame(fold_rows)
print('OOF:',len(oof),'| evidence:',len(evidence))
display(fold_audit)

## 5. 성능 비교 및 구성요소별 효과

Top-3 후보 생성 자체의 성능과 후보 내 재순위화 성능을 구분합니다. Macro-F1과 함께 클래스별 Recall 및 F1을 확인합니다.

In [ ]:
def metric_row(method,pred_col):
    return {'method':method,'accuracy':accuracy_score(oof['true_hs'],oof[pred_col]),'macro_f1':f1_score(oof['true_hs'],oof[pred_col],average='macro',zero_division=0),'weighted_f1':f1_score(oof['true_hs'],oof[pred_col],average='weighted',zero_division=0)}
metric_rows = [metric_row('SVM baseline','svm_top1')]
for method in METHODS: metric_rows.append(metric_row(method,method+'_top1'))
metrics = pd.DataFrame(metric_rows)
oof['svm_top3_hit'] = oof.apply(lambda r:r['true_hs'] in r['svm_top3'].split(' | '),axis=1)
print('SVM Top-3 hit:',oof['svm_top3_hit'].mean())
display(metrics.sort_values('macro_f1',ascending=False))
change_rows = []
for method in METHODS:
    pred = oof[method+'_top1']
    change_rows.append({'method':method,'changed_from_svm':int((pred!=oof['svm_top1']).sum()),'corrected':int(((pred!=oof['svm_top1'])&(pred==oof['true_hs'])).sum()),'damaged':int(((pred!=oof['svm_top1'])&(oof['svm_top1']==oof['true_hs'])).sum())})
display(pd.DataFrame(change_rows))
class_rows = []
for method in ['svm']+METHODS:
    pred_col = 'svm_top1' if method=='svm' else method+'_top1'
    for label in sorted(oof['true_hs'].unique()):
        mask = oof['true_hs'].eq(label)
        class_rows.append({'method':method,'hs_code':label,'n':int(mask.sum()),'recall':float((oof.loc[mask,pred_col]==label).mean()),'f1':float(f1_score(oof['true_hs'],oof[pred_col],labels=[label],average='macro',zero_division=0))})
class_metrics = pd.DataFrame(class_rows)
display(class_metrics.pivot(index='hs_code',columns='method',values='f1').round(3))

## 6. 검색 근거 진단

In [ ]:
if len(evidence):
    display(evidence['evidence_kind'].value_counts().rename_axis('evidence_kind').to_frame('count'))
    display(evidence.groupby('evidence_kind')['similarity'].agg(['count','mean','max']).round(4))
    display(evidence[evidence['evidence_kind'].eq('sbert_case_ruling')].sort_values(['fold','row_index','similarity'],ascending=[True,True,False]).head(20))
display(oof.head(10))

## 7. 결과 저장

- 11_hybrid_predictions.csv: 제품별 SVM 및 4개 재순위화 결과
- 11_hybrid_metrics.csv: 방법별 Accuracy, Macro-F1, Weighted-F1
- 11_hybrid_class_metrics.csv: 클래스별 Recall 및 F1
- 11_hybrid_evidence.csv: 지식베이스 및 과거 판정요지 근거
- 11_fold_audit.csv: Fold별 데이터 누수 점검
- 11_hybrid_change_analysis.csv: SVM 대비 예측 변경 진단

In [ ]:
change_analysis = pd.DataFrame(change_rows)
oof.to_csv('11_hybrid_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv('11_hybrid_metrics.csv',index=False,encoding='utf-8-sig')
class_metrics.to_csv('11_hybrid_class_metrics.csv',index=False,encoding='utf-8-sig')
evidence.to_csv('11_hybrid_evidence.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv('11_fold_audit.csv',index=False,encoding='utf-8-sig')
change_analysis.to_csv('11_hybrid_change_analysis.csv',index=False,encoding='utf-8-sig')
print('11번 결과 저장 완료')